# GitMind Commit Model — Dataset Exploration

Exploration du dataset CommitChronicle avant la préparation du corpus
d'entraînement de notre modèle de génération de messages de commit.

**Objectif :** analyser un échantillon des trois partitions sans télécharger
le dataset complet.

Dataset : [JetBrains-Research/commit-chronicle](https://huggingface.co/datasets/JetBrains-Research/commit-chronicle)


In [ ]:
%pip -q install "datasets>=4,<5" "pyyaml" "transformers==4.57.6" "tokenizers==0.22.2" "sentencepiece>=0.1.91,!=0.1.92"

In [ ]:
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/gitmind-commit-model")

if not REPO_DIR.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/EgueRichnel2025/gitmind-commit-model.git",
            str(REPO_DIR),
        ],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        check=True,
    )

%cd /content/gitmind-commit-model

In [ ]:
import json
import re
from collections import Counter
from itertools import islice
from pathlib import Path

import yaml
from datasets import load_dataset

with open("configs/data.yaml", encoding="utf-8") as file:
    CONFIG = yaml.safe_load(file)

DATASET_ID = CONFIG["primary"]["repository"]
DATASET_CONFIG = CONFIG["primary"]["configuration"]
SAMPLE_SIZE = CONFIG["exploration"]["sample_size_per_split"]
SEED = CONFIG["exploration"]["random_seed"]

print("Dataset :", DATASET_ID)
print("Configuration :", DATASET_CONFIG)
print("Taille de chaque échantillon :", SAMPLE_SIZE)

In [ ]:
def load_sample(split_name):
    print(f"Chargement de la partition : {split_name}")

    stream = load_dataset(
        DATASET_ID,
        DATASET_CONFIG,
        split=split_name,
        streaming=True,
    )

    print("Structure :", stream.features)

    stream = stream.shuffle(
        seed=SEED,
        buffer_size=10_000,
    )

    records = list(islice(iter(stream), SAMPLE_SIZE))

    print(f"Exemples récupérés : {len(records)}")
    return records


SAMPLES = {}

for split_name in ("train", "validation", "test"):
    SAMPLES[split_name] = load_sample(split_name)

In [ ]:
SENSITIVE_PATTERNS = {
    "email": re.compile(
        r"\\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\\.[A-Z]{2,}\\b",
        re.IGNORECASE,
    ),
    "ipv4": re.compile(r"\\b(?:[0-9]{1,3}\\.){3}[0-9]{1,3}\\b"),
    "github_token": re.compile(r"\\bgh[pousr]_[A-Za-z0-9]{20,}\\b"),
    "aws_key": re.compile(r"\\bAKIA[0-9A-Z]{16}\\b"),
    "private_key": re.compile(r"-----BEGIN (?:RSA |EC |OPENSSH )?PRIVATE KEY-----"),
}


def distribution(values):
    values = sorted(values)

    if not values:
        return {}

    def percentile(fraction):
        index = round((len(values) - 1) * fraction)
        return values[index]

    return {
        "count": len(values),
        "min": values[0],
        "median": percentile(0.50),
        "p90": percentile(0.90),
        "p95": percentile(0.95),
        "max": values[-1],
    }


def analyze_split(records):
    languages = Counter()
    licenses = Counter()
    seen_commits = set()

    diff_lengths = []
    message_lengths = []
    modification_counts = []

    empty_diffs = 0
    empty_messages = 0
    duplicate_commits = 0
    same_as_original = 0
    sensitive_counts = Counter()

    previews = []

    for row in records:
        message = str(row.get("message") or "").strip()
        original = str(row.get("original_message") or "").strip()
        modifications = row.get("mods") or []

        diffs = [str(mod.get("diff") or "") for mod in modifications if isinstance(mod, dict)]

        diff_text = "\\n".join(diffs).strip()

        language = str(row.get("language") or "<missing>")
        license_name = str(row.get("license") or "<missing>")
        repo = str(row.get("repo") or "<missing>")
        commit_hash = str(row.get("hash") or "")

        languages[language] += 1
        licenses[license_name] += 1

        if commit_hash:
            commit_id = (repo, commit_hash)
            if commit_id in seen_commits:
                duplicate_commits += 1
            seen_commits.add(commit_id)

        diff_lengths.append(len(diff_text))
        message_lengths.append(len(message))
        modification_counts.append(len(modifications))

        if not diff_text:
            empty_diffs += 1

        if not message:
            empty_messages += 1

        if message and message == original:
            same_as_original += 1

        for name, pattern in SENSITIVE_PATTERNS.items():
            if pattern.search(diff_text):
                sensitive_counts[name] += 1

        # Never include raw commit messages in the exploration report.
        if len(previews) < 5:
            previews.append(
                {
                    "repository": repo,
                    "language": language,
                    "license": license_name,
                    "message_characters": len(message),
                    "modified_files": len(modifications),
                    "diff_characters": len(diff_text),
                }
            )

    return {
        "sampled_examples": len(records),
        "empty_diffs": empty_diffs,
        "empty_messages": empty_messages,
        "duplicate_commits": duplicate_commits,
        "message_equals_original": same_as_original,
        "diff_characters": distribution(diff_lengths),
        "message_characters": distribution(message_lengths),
        "modified_files_per_commit": distribution(modification_counts),
        "languages": dict(languages.most_common(20)),
        "licenses": dict(licenses.most_common(20)),
        "records_with_potential_sensitive_patterns": dict(sensitive_counts),
        "examples_preview": previews,
    }

In [ ]:
REPORT = {
    "dataset": DATASET_ID,
    "configuration": DATASET_CONFIG,
    "sample_size_requested": SAMPLE_SIZE,
    "seed": SEED,
    "splits": {},
}

for split_name, records in SAMPLES.items():
    print(f"\\n{'=' * 65}")
    print(f"ANALYSE : {split_name.upper()}")
    print("=" * 65)

    REPORT["splits"][split_name] = analyze_split(records)

    print(
        json.dumps(
            REPORT["splits"][split_name],
            indent=2,
            ensure_ascii=False,
        )
    )

output_path = Path("data/interim/commit_chronicle_exploration.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

output_path.write_text(
    json.dumps(REPORT, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("\\nRapport enregistré :", output_path)

## Résultat attendu

À la fin de cette exploration, nous disposerons de statistiques pour les
trois partitions.

Ces résultats serviront à déterminer les règles de nettoyage, les longueurs
maximales des entrées et des sorties, ainsi que le filtrage des licences.

**Important :** ces statistiques portent sur les échantillons examinés,
pas sur l'intégralité des 10,7 millions de commits.


In [ ]:
from google.colab import files

files.download("data/interim/commit_chronicle_exploration.json")

# CodeT5 Tokenizer Audit

This section measures the actual token lengths of Git diffs and commit messages.

The tokenizer is CodeT5-small's RobertaTokenizerFast.

No raw commit messages or personal information are printed.


In [ ]:
import tokenizers
import transformers
from transformers import AutoTokenizer

MODEL_NAME = "Salesforce/codet5-small"

print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

MODEL_LIMIT = 512
TARGET_LIMIT = 128

print("Tokenizer:", type(tokenizer).__name__)
print("Model input limit:", MODEL_LIMIT)
print("Target limit:", TARGET_LIMIT)

In [ ]:
def serialize_commit(row):
    sections = []

    for modification in row.get("mods") or []:
        old_path = modification.get("old_path")
        new_path = modification.get("new_path")
        change_type = modification.get("change_type", "UNKNOWN")
        diff = modification.get("diff") or ""

        if not diff.strip():
            continue

        if old_path and new_path and old_path != new_path:
            file_path = f"{old_path} -> {new_path}"
        else:
            file_path = new_path or old_path or "unknown"

        sections.append(f"### FILE: {file_path}\\n### CHANGE: {change_type}\\n{diff}")

    return "\\n\\n".join(sections)


def token_length(text):
    if not text:
        return 0

    # Use the fast tokenizer backend directly to avoid
    # misleading warnings during diagnostic measurements.
    return len(
        tokenizer.backend_tokenizer.encode(
            text,
            add_special_tokens=True,
        ).ids
    )


def summarize_lengths(values):
    values = np.asarray(values)

    return {
        "count": int(len(values)),
        "median": round(float(np.percentile(values, 50)), 2),
        "p90": round(float(np.percentile(values, 90)), 2),
        "p95": round(float(np.percentile(values, 95)), 2),
        "p99": round(float(np.percentile(values, 99)), 2),
        "maximum": int(values.max()),
    }


TOKEN_REPORT = {}

for split_name, records in SAMPLES.items():
    source_lengths = []
    target_lengths = []
    original_target_lengths = []

    for row in records:
        source_lengths.append(token_length(serialize_commit(row)))

        target_lengths.append(token_length(str(row.get("message") or "")))

        original_target_lengths.append(token_length(str(row.get("original_message") or "")))

    source_lengths = np.asarray(source_lengths)
    target_lengths = np.asarray(target_lengths)
    original_target_lengths = np.asarray(original_target_lengths)

    TOKEN_REPORT[split_name] = {
        "input_diff_tokens": summarize_lengths(source_lengths),
        "target_message_tokens": summarize_lengths(target_lengths),
        "original_message_tokens": summarize_lengths(original_target_lengths),
        "input_over_512": int(np.sum(source_lengths > MODEL_LIMIT)),
        "input_truncation_rate_percent": round(
            float(np.mean(source_lengths > MODEL_LIMIT) * 100), 2
        ),
        "target_over_128": int(np.sum(target_lengths > TARGET_LIMIT)),
        "target_over_128_rate_percent": round(
            float(np.mean(target_lengths > TARGET_LIMIT) * 100), 2
        ),
    }

    print(f"\\n{split_name.upper()}")
    print(
        json.dumps(
            TOKEN_REPORT[split_name],
            indent=2,
            ensure_ascii=False,
        )
    )

In [ ]:
for split_name, records in SAMPLES.items():
    different = 0
    processed_shorter = 0
    processed_longer = 0
    equal_length = 0

    for row in records:
        message = str(row.get("message") or "")
        original = str(row.get("original_message") or "")

        if message != original:
            different += 1

        processed_count = token_length(message)
        original_count = token_length(original)

        if processed_count < original_count:
            processed_shorter += 1
        elif processed_count > original_count:
            processed_longer += 1
        else:
            equal_length += 1

    print(f"\\n{split_name.upper()}")
    print("Different messages:", different)
    print("Processed message shorter:", processed_shorter)
    print("Processed message longer:", processed_longer)
    print("Equal token lengths:", equal_length)

## Privacy and reproducibility

Notebook outputs are intentionally cleared before publication.

The exploration report contains aggregate statistics and metadata,
not raw commit messages.

The tokenizer audit can be reproduced by executing the notebook cells
in order.
